# 05 — Creación de PostGIS y carga cruda del piloto MT Montecarlo

Este notebook crea una base local PostgreSQL/PostGIS para el piloto de media tensión de Montecarlo y carga únicamente los TSV crudos necesarios para la primera etapa.

La meta de esta práctica es dejar una base trazable y auditable. Todavía **no** se construyen geometrías finales: la reconstrucción desde `TMP_SHAPEFILE.Datos_Objeto` y sus tokens `§` corresponde a una fase posterior.

## 1. Objetivo y alcance

En esta etapa se trabaja con alcance controlado:

| Tema | Decisión |
|---|---|
| Servicio Docker | `postgis` |
| Imagen | `postgis/postgis:16-3.4` |
| Base local | `ceml_gis` |
| Esquemas | `crudo`, `depuracion`, `gis`, `auditoria` |
| Carga inicial | TSV del piloto MT como columnas `TEXT` |
| Fuera de esta notebook | Geometrías finales, BT, suministros, alumbrado, reclamos y catastro completo |

Los TSV esperados son `tmp_shapefile.tsv`, `objetos_red.tsv`, `set.tsv`, `seccionadores.tsv` y `mt_cables.tsv`. Si faltan, la notebook lo informa y se puede volver a ejecutar la sección de exportación del notebook 01.

## 2. Preparación de rutas y parámetros

Primero ubicamos la raíz del proyecto y definimos los valores de conexión local. Las variables de entorno permiten cambiar credenciales sin modificar la notebook, pero se muestran los valores por defecto usados en el laboratorio.

In [1]:
# pathlib permite resolver rutas sin depender de la carpeta exacta desde donde se abrió Jupyter.
from pathlib import Path

# os permite leer variables de entorno para parametrizar la conexión local.
import os

# csv permite leer encabezados TSV sin convertir tipos de datos automáticamente.
import csv

# psycopg es el driver moderno de PostgreSQL para Python.
# Lo importamos con control de error porque el kernel puede no tener instaladas aún las dependencias nuevas.
try:
    import psycopg
    from psycopg import sql
    PSYCOPG_DISPONIBLE = True
except ModuleNotFoundError:
    psycopg = None
    sql = None
    PSYCOPG_DISPONIBLE = False
    print('No está instalado el paquete psycopg en este kernel.')
    print('Ejecutar desde la raíz del proyecto: python3 -m pip install -r requirements.txt')
    print('Después de instalar, reiniciar el kernel de Jupyter y volver a ejecutar la notebook.')

# Buscamos la raíz del proyecto para que el notebook funcione desde JupyterLab o desde la carpeta notebooks.
RAIZ = Path.cwd()
while RAIZ.name != 'ceml_gis' and RAIZ.parent != RAIZ:
    RAIZ = RAIZ.parent

# Definimos la carpeta donde el notebook 01 deja los TSV exportados desde SQL Server.
EXPORTS = RAIZ / 'sqlserver' / 'exports'

# Estos valores coinciden con el servicio postgis definido en docker-compose.yml.
DB_CONFIG = {
    'host': os.getenv('POSTGRES_HOST', 'localhost'),
    'port': int(os.getenv('POSTGRES_PORT', '5439')),
    'dbname': os.getenv('POSTGRES_DB', 'ceml_gis'),
    'user': os.getenv('POSTGRES_USER', 'ceml'),
    'password': os.getenv('POSTGRES_PASSWORD', 'ceml_admin_2026'),
}

# Listamos solo las tablas del piloto MT para evitar ampliar el alcance sin validación.
TSV_PILOTO = {
    'tmp_shapefile': EXPORTS / 'tmp_shapefile.tsv',
    'objetos_red': EXPORTS / 'objetos_red.tsv',
    'set': EXPORTS / 'set.tsv',
    'seccionadores': EXPORTS / 'seccionadores.tsv',
    'mt_cables': EXPORTS / 'mt_cables.tsv',
}

# Esta función corta con un mensaje claro si se intenta conectar sin tener instalado psycopg.
def exigir_psycopg() -> None:
    if not PSYCOPG_DISPONIBLE:
        raise RuntimeError(
            'Falta instalar psycopg en este kernel. '
            'Ejecutar: python -m pip install -r requirements.txt, '
            'reiniciar el kernel y volver a correr la notebook.'
        )

# Mostramos el contexto de trabajo para que el estudiante confirme rutas y conexión.
print('Raíz del proyecto:', RAIZ)
print('Carpeta de TSV:', EXPORTS)
print('Servicio Docker esperado: postgis')
print('Base local:', DB_CONFIG['dbname'])
print('Usuario local:', DB_CONFIG['user'])
print('Puerto local:', DB_CONFIG['port'])


Raíz del proyecto: /home/martin/Code/ceml_gis
Carpeta de TSV: /home/martin/Code/ceml_gis/sqlserver/exports
Servicio Docker esperado: postgis
Base local: ceml_gis
Usuario local: ceml
Puerto local: 5439


## 3. Levantar PostGIS con Docker Compose

La siguiente celda levanta solo el servicio `postgis` desde la raíz del proyecto. Si el contenedor ya existe, Docker Compose lo reutiliza.

In [2]:
# Este comando inicia únicamente el servicio postgis definido en docker-compose.yml.
# Se ejecuta desde la raíz para que Docker encuentre rutas relativas como ./postgis/data.
!cd "{RAIZ}" && docker compose up -d postgis


[+] up 1/1
 ✔ Container ceml-postgis Running                                           0.0s


## 4. Verificar conectividad y extensión PostGIS

Ahora probamos la conexión desde Python. Si esta celda falla, revisar que Docker esté activo y que el servicio `postgis` haya terminado su inicialización.

In [3]:
# Antes de conectarnos, verificamos que el driver esté instalado en el kernel actual.
exigir_psycopg()

# Abrimos una conexión corta para validar que PostgreSQL responde.
# autocommit=True permite ejecutar CREATE EXTENSION fuera de una transacción larga.
with psycopg.connect(**DB_CONFIG, autocommit=True) as conn:
    with conn.cursor() as cur:
        # Activamos PostGIS si todavía no estaba instalado en la base.
        cur.execute('CREATE EXTENSION IF NOT EXISTS postgis;')
        
        # Consultamos la versión para confirmar que la extensión está operativa.
        cur.execute('SELECT PostGIS_Full_Version();')
        version_postgis = cur.fetchone()[0]

# Mostramos una confirmación clara para la clase.
print('Conexión a PostGIS verificada correctamente.')
print('Versión PostGIS:')
print(version_postgis)


Conexión a PostGIS verificada correctamente.
Versión PostGIS:
POSTGIS="3.4.3 e365945" [EXTENSION] PGSQL="160" GEOS="3.9.0-CAPI-1.16.2" PROJ="7.2.1 NETWORK_ENABLED=OFF URL_ENDPOINT=https://cdn.proj.org USER_WRITABLE_DIRECTORY=/var/lib/postgresql/.local/share/proj DATABASE_PATH=/usr/share/proj/proj.db" LIBXML="2.9.10" LIBJSON="0.15" LIBPROTOBUF="1.3.3" WAGYU="0.5.0 (Internal)" TOPOLOGY


## 5. Crear esquemas de trabajo

Se crean cuatro esquemas con responsabilidades separadas. En esta notebook se carga solo `crudo`; los otros esquemas quedan preparados para depuración, publicación GIS y auditoría.

In [4]:
# Definimos los esquemas acordados para ordenar el flujo de trabajo.
ESQUEMAS = ['crudo', 'depuracion', 'gis', 'auditoria']

# Antes de conectarnos, verificamos que el driver esté instalado en el kernel actual.
exigir_psycopg()

# Creamos los esquemas si no existen y luego listamos los esquemas relevantes.
with psycopg.connect(**DB_CONFIG, autocommit=True) as conn:
    with conn.cursor() as cur:
        for esquema in ESQUEMAS:
            cur.execute(sql.SQL('CREATE SCHEMA IF NOT EXISTS {};').format(sql.Identifier(esquema)))
        
        # Esta consulta confirma que los cuatro esquemas quedaron disponibles.
        cur.execute(
            """
            SELECT schema_name
            FROM information_schema.schemata
            WHERE schema_name = ANY(%s)
            ORDER BY schema_name;
            """,
            (ESQUEMAS,),
        )
        esquemas_creados = [fila[0] for fila in cur.fetchall()]

# Mostramos el resultado como una lista simple.
print('Esquemas disponibles:', ', '.join(esquemas_creados))


Esquemas disponibles: auditoria, crudo, depuracion, gis


## 6. Preparar auditoría de carga

Cada archivo procesado deja un registro de auditoría. Esto permite distinguir entre archivos cargados, archivos faltantes y errores de lectura sin perder trazabilidad.

In [5]:
# Esta tabla registra el resultado de cada intento de carga del piloto MT.
# No reemplaza auditorías de geometría; solo audita la carga cruda de TSV.
SQL_AUDITORIA = """
CREATE TABLE IF NOT EXISTS auditoria.cargas_piloto_mt (
    id bigserial PRIMARY KEY,
    fecha_carga timestamptz NOT NULL DEFAULT now(),
    archivo text NOT NULL,
    tabla_destino text NOT NULL,
    filas integer NOT NULL DEFAULT 0,
    estado text NOT NULL,
    notas text NOT NULL DEFAULT ''
);
"""

# Antes de conectarnos, verificamos que el driver esté instalado en el kernel actual.
exigir_psycopg()

# Creamos la tabla de auditoría antes de intentar cargar cualquier TSV.
with psycopg.connect(**DB_CONFIG, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute(SQL_AUDITORIA)

# Informamos que la auditoría quedó preparada.
print('Tabla de auditoría preparada: auditoria.cargas_piloto_mt')


Tabla de auditoría preparada: auditoria.cargas_piloto_mt


## 7. Revisar disponibilidad de TSV

Antes de cargar, revisamos qué archivos existen. Si alguno falta, no se detiene toda la notebook: se registra en auditoría y se informa que debe repetirse la exportación del notebook 01.

In [6]:
# Construimos una tabla de estado para los cinco TSV esperados del piloto MT.
estado_archivos = []
for tabla, path in TSV_PILOTO.items():
    estado_archivos.append({
        'tabla': tabla,
        'archivo': path.name,
        'existe': path.exists(),
        'ruta': str(path.relative_to(RAIZ)) if path.exists() else str(path),
    })

# Mostramos mensajes claros para que el estudiante sepa qué hacer si faltan insumos.
for item in estado_archivos:
    if item['existe']:
        print(f"Disponible: {item['archivo']} -> crudo.{item['tabla']}")
    else:
        print(f"Falta {item['archivo']}. Volver a ejecutar la sección de exportación del notebook 01.")


Disponible: tmp_shapefile.tsv -> crudo.tmp_shapefile
Disponible: objetos_red.tsv -> crudo.objetos_red
Disponible: set.tsv -> crudo.set
Disponible: seccionadores.tsv -> crudo.seccionadores
Disponible: mt_cables.tsv -> crudo.mt_cables


## 8. Cargar TSV crudos como texto

La carga crea una tabla por archivo en el esquema `crudo`. Todas las columnas se crean como `TEXT` para preservar códigos, identificadores CAD, valores vacíos y textos originales.

Si una tabla ya existe, se reemplaza para que la carga sea reproducible durante la práctica.

In [7]:
# Esta función normaliza nombres de columnas vacías o repetidas sin interpretar los datos.
def normalizar_columnas(columnas: list[str]) -> list[str]:
    columnas_limpias = []
    usados = set()
    for indice, columna in enumerate(columnas, start=1):
        nombre = columna.strip() or f'columna_{indice}'
        base = nombre
        contador = 2
        while nombre.lower() in usados:
            nombre = f'{base}_{contador}'
            contador += 1
        usados.add(nombre.lower())
        columnas_limpias.append(nombre)
    return columnas_limpias

# Esta función inserta una fila de auditoría por cada resultado de carga.
def registrar_auditoria(cur, archivo: str, tabla_destino: str, filas: int, estado: str, notas: str) -> None:
    cur.execute(
        """
        INSERT INTO auditoria.cargas_piloto_mt (archivo, tabla_destino, filas, estado, notas)
        VALUES (%s, %s, %s, %s, %s);
        """,
        (archivo, tabla_destino, filas, estado, notas),
    )

# Esta función carga un TSV existente en crudo usando COPY para preservar el contenido textual.
def cargar_tsv_crudo(conn, tabla: str, path: Path) -> dict:
    tabla_destino = f'crudo.{tabla}'
    with conn.cursor() as cur:
        if not path.exists():
            notas = 'Archivo no encontrado. Volver a ejecutar la sección de exportación del notebook 01.'
            registrar_auditoria(cur, path.name, tabla_destino, 0, 'no_encontrado', notas)
            return {'tabla': tabla_destino, 'archivo': path.name, 'filas': 0, 'estado': 'no_encontrado', 'notas': notas}

        try:
            # Leemos solo la primera fila para obtener encabezados y crear columnas TEXT.
            with path.open('r', encoding='utf-8-sig', newline='') as archivo:
                lector = csv.reader(archivo, delimiter='\t')
                encabezado = next(lector, None)

            if not encabezado:
                notas = 'Archivo vacío o sin encabezado.'
                registrar_auditoria(cur, path.name, tabla_destino, 0, 'error', notas)
                return {'tabla': tabla_destino, 'archivo': path.name, 'filas': 0, 'estado': 'error', 'notas': notas}

            columnas = normalizar_columnas(encabezado)

            # Reemplazamos la tabla cruda para que repetir la carga no duplique filas.
            cur.execute(sql.SQL('DROP TABLE IF EXISTS {}.{};').format(sql.Identifier('crudo'), sql.Identifier(tabla)))
            definicion_columnas = sql.SQL(', ').join(
                sql.SQL('{} TEXT').format(sql.Identifier(columna)) for columna in columnas
            )
            cur.execute(
                sql.SQL('CREATE TABLE {}.{} ({});').format(
                    sql.Identifier('crudo'),
                    sql.Identifier(tabla),
                    definicion_columnas,
                )
            )

            # COPY carga el TSV completo respetando el encabezado y el delimitador tabulación.
            consulta_copy = sql.SQL(
                "COPY {}.{} ({}) FROM STDIN WITH (FORMAT csv, HEADER true, DELIMITER E'\\t', QUOTE '\"')"
            ).format(
                sql.Identifier('crudo'),
                sql.Identifier(tabla),
                sql.SQL(', ').join(sql.Identifier(columna) for columna in columnas),
            )
            with cur.copy(consulta_copy) as copia:
                with path.open('r', encoding='utf-8-sig', newline='') as archivo:
                    for linea in archivo:
                        copia.write(linea)

            # Contamos filas ya cargadas en la tabla destino para auditar el resultado real.
            cur.execute(sql.SQL('SELECT count(*) FROM {}.{};').format(sql.Identifier('crudo'), sql.Identifier(tabla)))
            filas = cur.fetchone()[0]
            notas = f'Carga textual completa con {len(columnas)} columnas.'
            registrar_auditoria(cur, path.name, tabla_destino, filas, 'cargado', notas)
            return {'tabla': tabla_destino, 'archivo': path.name, 'filas': filas, 'estado': 'cargado', 'notas': notas}
        except Exception as exc:
            # Registramos el error para no perder trazabilidad si un TSV tiene formato inesperado.
            conn.rollback()
            with conn.cursor() as cur_error:
                registrar_auditoria(cur_error, path.name, tabla_destino, 0, 'error', str(exc))
            return {'tabla': tabla_destino, 'archivo': path.name, 'filas': 0, 'estado': 'error', 'notas': str(exc)}

# Antes de conectarnos, verificamos que el driver esté instalado en el kernel actual.
exigir_psycopg()

# Ejecutamos la carga de los cinco TSV esperados del piloto MT.
resultados_carga = []
with psycopg.connect(**DB_CONFIG) as conn:
    for tabla, path in TSV_PILOTO.items():
        resultado = cargar_tsv_crudo(conn, tabla, path)
        resultados_carga.append(resultado)
        # Confirmamos cada archivo por separado para que un error posterior no revierta cargas anteriores.
        conn.commit()

# Mostramos el resultado resumido de la carga.
for resultado in resultados_carga:
    print(f"{resultado['tabla']}: {resultado['estado']} ({resultado['filas']} filas) - {resultado['notas']}")


crudo.tmp_shapefile: cargado (125688 filas) - Carga textual completa con 6 columnas.
crudo.objetos_red: cargado (69134 filas) - Carga textual completa con 18 columnas.
crudo.set: error (0 filas) - unquoted carriage return found in data
HINT:  Use quoted CSV field to represent carriage return.
CONTEXT:  COPY set, line 739
crudo.seccionadores: cargado (1063 filas) - Carga textual completa con 21 columnas.
crudo.mt_cables: cargado (40 filas) - Carga textual completa con 3 columnas.


## 9. Validaciones posteriores

Estas consultas comprueban que PostGIS responde, que los esquemas existen y que las tablas crudas cargadas tienen filas. También se revisan los registros de auditoría más recientes.

In [8]:
# Antes de conectarnos, verificamos que el driver esté instalado en el kernel actual.
exigir_psycopg()

# Ejecutamos validaciones de lectura para confirmar el estado de la base después de la carga.
with psycopg.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        # Confirmamos versión resumida de PostGIS.
        cur.execute('SELECT PostGIS_Version();')
        version_resumida = cur.fetchone()[0]

        # Listamos los esquemas de trabajo creados.
        cur.execute(
            """
            SELECT schema_name
            FROM information_schema.schemata
            WHERE schema_name = ANY(%s)
            ORDER BY schema_name;
            """,
            (ESQUEMAS,),
        )
        esquemas = [fila[0] for fila in cur.fetchall()]

        # Contamos filas de las tablas crudas que efectivamente existen.
        conteos = []
        for tabla in TSV_PILOTO:
            cur.execute(
                """
                SELECT EXISTS (
                    SELECT 1
                    FROM information_schema.tables
                    WHERE table_schema = 'crudo' AND table_name = %s
                );
                """,
                (tabla,),
            )
            existe = cur.fetchone()[0]
            if existe:
                cur.execute(sql.SQL('SELECT count(*) FROM {}.{};').format(sql.Identifier('crudo'), sql.Identifier(tabla)))
                filas = cur.fetchone()[0]
                conteos.append((f'crudo.{tabla}', filas))

        # Recuperamos las últimas auditorías para revisar qué pasó en esta ejecución.
        cur.execute(
            """
            SELECT archivo, tabla_destino, filas, estado, notas
            FROM auditoria.cargas_piloto_mt
            ORDER BY id DESC
            LIMIT 10;
            """
        )
        auditorias = cur.fetchall()

# Presentamos los resultados con etiquetas claras para lectura en clase.
print('PostGIS versión:', version_resumida)
print('Esquemas:', ', '.join(esquemas))
print('Conteos en tablas crudas:')
for tabla, filas in conteos:
    print(f'- {tabla}: {filas} filas')
print('Auditorías recientes:')
for archivo, tabla_destino, filas, estado, notas in auditorias:
    print(f'- {archivo} -> {tabla_destino}: {estado}, {filas} filas, {notas}')


PostGIS versión: 3.4 USE_GEOS=1 USE_PROJ=1 USE_STATS=1
Esquemas: auditoria, crudo, depuracion, gis
Conteos en tablas crudas:
- crudo.tmp_shapefile: 125688 filas
- crudo.objetos_red: 69134 filas
- crudo.seccionadores: 1063 filas
- crudo.mt_cables: 40 filas
Auditorías recientes:
- mt_cables.tsv -> crudo.mt_cables: cargado, 40 filas, Carga textual completa con 3 columnas.
- seccionadores.tsv -> crudo.seccionadores: cargado, 1063 filas, Carga textual completa con 21 columnas.
- set.tsv -> crudo.set: error, 0 filas, unquoted carriage return found in data
HINT:  Use quoted CSV field to represent carriage return.
CONTEXT:  COPY set, line 739
- objetos_red.tsv -> crudo.objetos_red: cargado, 69134 filas, Carga textual completa con 18 columnas.
- tmp_shapefile.tsv -> crudo.tmp_shapefile: cargado, 125688 filas, Carga textual completa con 6 columnas.


## 10. Próxima fase: reconstrucción geométrica

La base queda lista para trabajar con datos crudos del piloto MT. La siguiente fase debe analizar `crudo.tmp_shapefile`, separar los tokens `§` de `Datos_Objeto`, clasificar entidades CAD y construir geometrías candidatas en `depuracion`.

No conviene publicar en `gis` hasta validar tipos geométricos, coordenadas suficientes, cobertura de atributos y rechazos auditados.